# Sprint 1 - full pipeline in one notebook
**Feed documents in -> OCR (Squad 1) -> human review if needed -> PII masking (Squad 2) -> clinical extraction (Squad 3).**
No squad code is changed or copied: this notebook only *runs* it, in order, against the same Supabase database.

**Before you start**
1. `Runtime -> Change runtime type -> T4 GPU` (Squad 1's PaddleOCR needs it).
2. Colab Secrets (key icon, left): add `SUPABASE_URL`, `SUPABASE_SERVICE_KEY` and `GEMINI_API_KEY`, and switch **Notebook access** on for each.
3. Run cells top to bottom the first time. After that you normally only use **Step 4** (new case), **Step 5** (reviews) and **Step 6** (status).

If Colab asks to restart the runtime after Step 1, restart, then continue from Step 2.

## Step 1 - GPU check and installs (same packages the squads list)

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
assert gpu.returncode == 0, "No GPU found. Runtime > Change runtime type > T4 GPU, then run this cell again."
print(gpu.stdout.strip())

In [ ]:
# Squad 1 (OCR_reader/requirements.txt.txt) - paddlepaddle-gpu must come from the CUDA 12.6 index
!pip install -q paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q -U paddleocr "nvidia-nccl-cu12>=2.27.3"
!apt-get update -qq          # Colab's package list is often stale; without this poppler can 404
!apt-get install -y -qq poppler-utils
!pip install -q pdf2image opencv-python-headless supabase "httpx==0.28.1" "pydantic==2.14.1"
# Squad 2 and Squad 3
!pip install -q Pillow google-genai pymupdf
!python -c "import paddle; paddle.utils.run_check()"
!pdftoppm -v                 # must print a version; poppler turns PDF pages into images
# Expected and harmless: pip may warn that paddlepaddle-gpu wants nvidia-nccl-cu12==2.25.1 (Squad 1's own install upgrades it)


## Step 2 - Get the project and connect to Supabase
Either set `REPO_URL` to your Git repository (private repos need a token in the URL), **or** leave it empty and upload `Sprint-1-main.zip` (the project *including* the `orchestrator/` folder) when asked.

In [ ]:
import os, sys, subprocess, zipfile
from pathlib import Path

REPO_URL = ""          # e.g. "https://github.com/<org>/Sprint-1.git"; empty = upload the zip instead
WORK = Path("/content/project")

if not WORK.exists():
    WORK.mkdir(parents=True)
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, str(WORK)], check=True)
    else:
        from google.colab import files
        up = files.upload()                                   # choose the project zip
        zname = next(iter(up))
        with zipfile.ZipFile(zname) as z:
            z.extractall(WORK)

# the project root is the folder that holds OCR_reader/ and orchestrator/
ROOT = next(p.parent for p in WORK.rglob("OCR_reader") if (p.parent / "orchestrator").exists())
sys.path.insert(0, str(ROOT / "orchestrator"))
print("Project root:", ROOT)

In [ ]:
from google.colab import userdata
for key in ("SUPABASE_URL", "SUPABASE_SERVICE_KEY", "GEMINI_API_KEY"):      # Squad 2/3 read these from the environment
    os.environ[key] = userdata.get(key)

from supabase import create_client
sb = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_SERVICE_KEY"])

# health check: both tables and both buckets must be reachable
sb.table("claims_cases").select("case_id").limit(1).execute()
sb.table("claim_documents").select("doc_id").limit(1).execute()
for b in ("raw-medical-docs", "processed-previews"):
    sb.storage.from_(b).list(path="", options={"limit": 1})
print("Supabase connected: tables and buckets OK")

## Step 3 - Build the pipeline

In [ ]:
import pipeline, intake, human_review, review_ui

ns = pipeline.load_squad1(ROOT, sb)        # Squad 1's own notebook cells, executed unchanged
P = pipeline.Pipeline(sb, ROOT, squad1_ns=ns)   # Squad 2 / 3 are run as `python db_runner.py` exactly as their READMEs say
print("Pipeline ready. (The OCR model loads on first use, so the first document takes longer.)")

## Step 4 - Feed a new case
Set the case number, run the cell, pick the scan(s) for that case (PDF / PNG / JPG). The files are validated, uploaded to Supabase, and the pipeline then runs by itself until every case is either **COMPLETED** or waiting for a person (human review, or a clearer scan from the hospital).

In [ ]:
from google.colab import files
CASE_NUMBER = "PA-2026-0001"                # <- edit: one case number per patient case

inbox = Path("/content/inbox") / CASE_NUMBER
inbox.mkdir(parents=True, exist_ok=True)
for name, data in files.upload().items():
    (inbox / name).write_bytes(data)

summary = intake.intake_case(sb, CASE_NUMBER, [inbox])
print(summary)
P.run_until_settled()

## Step 4b - Feed several patient folders at once (e.g. 5 patients x 4 documents)
Your folders can look like `Patient_01_Name/docs/*.pdf` (a `docs` sub-folder is used if it exists, so a sibling `preprocessed/` folder is never uploaded) or `Patient_01_Name/*.pdf`. Each patient folder becomes **one case**, numbered `Patient_01`, `Patient_02`, ... - the patient's name is never stored.

**Zip all patient folders into one zip** (or put them in Google Drive at `MyDrive/inbox/`), then run the two cells below. The first one only **previews**; nothing is written.

In [ ]:
import zipfile
from google.colab import files

SOURCE = "zip"                      # "zip" = upload a zip now;  "drive" = read MyDrive/inbox
if SOURCE == "zip":
    dest = Path("/content/patients"); dest.mkdir(exist_ok=True)
    for name in files.upload():
        zipfile.ZipFile(name).extractall(dest)
    INBOX = intake.find_inbox_root(dest)      # steps into a wrapper folder if the zip has one
else:
    from google.colab import drive
    drive.mount("/content/drive")
    INBOX = Path("/content/drive/MyDrive/inbox")

print("Preview (nothing written):")
for s in intake.intake_inbox(sb, INBOX, dry=True):
    print(f"  case {s['case_number']}: {len(s['uploaded'])} new, {len(s['skipped'])} already there, "
          f"ignored non-documents: {s['ignored'] or '-'}")

If the preview shows the cases and document counts you expect (e.g. 5 cases, 4 each), run this to upload and process everything:

In [ ]:
for s in intake.intake_inbox(sb, INBOX):
    print(s["case_number"], "->", len(s["uploaded"]), "uploaded")
P.run_until_settled()

## Step 5 - Human review
Cases that contain uncertain handwriting/print (calibrated confidence below the thresholds in the defence document) stop at **PENDING_HUMAN_REVIEW**. Run this cell, check each line against its cropped image, and press **Save decisions**. When a case has no lines left to review it continues through Squad 2 and Squad 3 automatically.

In [ ]:
review_ui.show(sb, P)

## Step 6 - Status

In [ ]:
pipeline.print_status(sb)

## Optional
**Answer an RFI** (the hospital sends a clearer scan for a case at `RFI_RESCAN_REQUIRED`): upload the new file with the *same file name* as the bad one and run the cell.

In [ ]:
from google.colab import files
CASE_NUMBER = "PA-2026-0001"                # <- the case waiting for a rescan
inbox = Path("/content/rescan") / CASE_NUMBER
inbox.mkdir(parents=True, exist_ok=True)
for name, data in files.upload().items():
    (inbox / name).write_bytes(data)
print(intake.intake_case(sb, CASE_NUMBER, [inbox], replace=True))
P.run_until_settled()

**Keep watching for new work** (stops after `max_minutes`, or press the stop button):

In [ ]:
P.watch(poll_seconds=30, max_minutes=60)